# Stem separation (BS-RoFormer)

Pick an audio file with the popup, separate it into stems with **BS-RoFormer**, and export the WAVs
with normalized names in the project root (`01-Mixer/`), ready for the app to load.

**Minimum hardware requirements**

- RAM: 8 GB minimum (16 GB recommended)
- CPU: any modern x86_64 works; on CPU separation takes several minutes per track
- GPU: optional (CUDA speeds it up significantly)
- Disk: ~3 GB free (PyTorch + ~700 MB model, downloaded automatically on first run)
- Python ≥ 3.10

Usage: run the cells top to bottom (`Shift+Enter`).

In [ ]:
import importlib.util, subprocess, sys

if importlib.util.find_spec('bs_roformer_infer') is None:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                           'bs-roformer-infer @ git+https://github.com/openmirlab/bs-roformer-infer.git'])
print('bs-roformer-infer ready')

In [ ]:
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()  # popup only, no main window
INPUT_FILE = filedialog.askopenfilename(
    title='Pick an audio file to separate',
    filetypes=[('Audio', '*.wav *.flac *.mp3 *.ogg *.m4a')])
root.destroy()

assert INPUT_FILE, 'No file selected'
print('Selected audio:', INPUT_FILE)

In [ ]:
import shutil
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent          # from the notebook (stems/) to the project root
WORK = Path('input'); WORK.mkdir(exist_ok=True)   # temp input folder for the CLI
shutil.copy(INPUT_FILE, WORK / Path(INPUT_FILE).name)

subprocess.run(['bs-roformer-infer', '--input_folder', str(WORK),
                '--store_dir', str(WORK / 'out')], check=True)

# Normalize names: lowercase <stem>.wav, no spaces, in the project root
STEMS = ['vocals', 'drums', 'bass', 'guitar', 'piano', 'other', 'instrumental']
exported = []
for f in (WORK / 'out').glob('*.wav'):
    label = next((s for s in STEMS if s in f.stem.lower()), None)
    if label:
        dest = PROJECT_ROOT / f'{label}.wav'
        shutil.move(str(f), dest)
        exported.append(dest.name)

shutil.rmtree(WORK)  # clean up temp files
print('Stems exported to', PROJECT_ROOT, ':', sorted(exported))